# K-ABENA Ch.14 — CIFAR-10 avec PyTorch

**Objectif** : reproduire les résultats du Tableau 14.1.1 du livre K-ABENA.

| Variante | Acc. cible | Gain cible |
|----------|-----------|------------|
| SGD standard | 93.2% | 0% |
| K-ABENA N=0 | 94.2% | 18.5% |
| K-ABENA N=0.4 | 94.6% | 14.2% |
| K-ABENA Adaptatif | 94.9% | 19.3% |
| Adam + K-ABENA | 95.1% | 17.8% |

**Coût K-ABENA vs SGD standard : +2 lignes dans la boucle.**

```python
# AVANT
loss = F.cross_entropy(model(X), y)

# APRÈS (+2 lignes)
losses = F.cross_entropy(model(X), y, reduction='none')
mask   = kabena_filter_torch(losses, K=K, N=N)
losses[mask].mean().backward()
```

In [ ]:
# !pip install kabena-ml[torch] torchvision -q
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import numpy as np
import time
from kabena_ml.core.filter import kabena_filter, calibrate_K
from kabena_ml.integrations.torch_utils import kabena_filter_torch

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS   = 200          # réduit à 10 pour démo rapide
DEMO     = True         # True = 10 époques, False = 200 (résultats complets)
if DEMO: EPOCHS = 10

SEEDS    = [42]         # [42, 7, 13, 99, 123] pour résultats complets
print(f'Device: {DEVICE} | Époques: {EPOCHS} | Mode: {"DÉMO" if DEMO else "COMPLET"}')

In [ ]:
# ── Données CIFAR-10 ──────────────────────────────────────────────────────
def get_cifar10_loaders(batch_size=128):
    mean, std = (0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)
    train_tfm = T.Compose([
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(mean, std),
    ])
    test_tfm = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
    train_ds = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=train_tfm)
    test_ds  = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=test_tfm)
    return (torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2),
            torch.utils.data.DataLoader(test_ds,  batch_size=256,        shuffle=False, num_workers=2))

train_loader, test_loader = get_cifar10_loaders()

In [ ]:
# ── Modèle ResNet-18 (identique au ch.14) ─────────────────────────────────
def get_resnet18(n_classes=10):
    model = torchvision.models.resnet18(weights=None)
    # Adapter pour CIFAR-10 (images 32×32 au lieu de 224×224)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()  # supprimer le maxpool
    model.fc      = nn.Linear(512, n_classes)
    return model.to(DEVICE)

In [ ]:
# ── KabenaScheduler intégré ───────────────────────────────────────────────
class KabenaScheduler:
    """Stratégie warm-up progressif (recommandée Ch.12)."""
    def __init__(self, strategy='warmup', q_init=5, q_target=20, T_warmup=20):
        self.strategy = strategy
        self.q_init, self.q_target, self.T_warmup = q_init, q_target, T_warmup
        self.K_prev = None

    def step(self, losses_np, epoch):
        if self.strategy == 'warmup':
            q = self.q_init + (self.q_target - self.q_init) * min(epoch / self.T_warmup, 1)
            return float(np.percentile(losses_np, q))
        elif self.strategy == 'percentile':
            return float(np.percentile(losses_np, self.q_target))
        return float(np.percentile(losses_np, 10))

In [ ]:
# ── Boucle d'entraînement universelle ─────────────────────────────────────
def train_one_epoch(model, loader, optimizer, scheduler_lr, K, N, variant, sched_K=None, epoch=0):
    model.train()
    total_m, total_n, losses_ep = 0, 0, []
    all_losses_ep1 = []

    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)

        # Forward — identique pour toutes les variantes
        logits = model(X)
        losses = F.cross_entropy(logits, y, reduction='none')  # ← toujours 'none'
        all_losses_ep1.extend(losses.detach().cpu().numpy())

        if variant == 'standard':
            # SGD / Adam standard : pas de filtre
            L = losses.mean()
            m = len(y)
        else:
            # K-ABENA : 1 ligne de différence
            K_batch = K if K is not None else sched_K.step(losses.detach().cpu().numpy(), epoch)
            mask = kabena_filter_torch(losses, K=K_batch, N=N)  # ← filtre
            m    = mask.sum().item()
            if m == 0:
                continue
            L = losses[mask].mean()

        optimizer.zero_grad()
        L.backward()
        optimizer.step()

        total_m += m
        total_n += len(y)
        losses_ep.append(L.item())

    scheduler_lr.step()
    gain = round((1 - total_m / total_n) * 100) if total_n > 0 else 0
    return np.mean(losses_ep), gain, np.array(all_losses_ep1)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        preds = model(X).argmax(1)
        correct += (preds == y).sum().item()
        total   += len(y)
    return correct / total * 100

In [ ]:
# ── Expérience principale ─────────────────────────────────────────────────
def run_experiment(variant, K=None, N=0.0, optimizer_cls=torch.optim.SGD, seed=42):
    """
    variant : 'standard' | 'ka_fixed_n0' | 'ka_fixed_n4' | 'adaptive' | 'adam_ka'
    K       : float (seuil fixe) ou None (calibrage auto à l'époque 1)
    N       : float [0,1] — proportion de mineures conservées
    """
    torch.manual_seed(seed); np.random.seed(seed)

    model = get_resnet18()
    if optimizer_cls == torch.optim.SGD:
        optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    # Learning rate schedule cosine (identique pour toutes les variantes)
    lr_sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    sched_K  = KabenaScheduler(strategy='warmup') if variant == 'adaptive' else None

    K_current = K  # sera calibré à l'époque 1 si None
    history   = []

    for epoch in range(EPOCHS):
        t0 = time.time()
        loss, gain, losses_ep1 = train_one_epoch(
            model, train_loader, optimizer, lr_sched,
            K_current, N, variant, sched_K, epoch
        )

        # Calibrage auto de K à la première époque
        if epoch == 0 and K_current is None and variant != 'standard':
            K_current = calibrate_K(losses_ep1, target_pct=0.10)
            print(f'  [Époque 1] K calibré automatiquement = {K_current:.4f}')

        acc = evaluate(model, test_loader)
        dt  = time.time() - t0
        history.append({'epoch': epoch, 'loss': loss, 'acc': acc, 'gain': gain, 'time': dt})

        if epoch % max(1, EPOCHS // 5) == 0 or epoch == EPOCHS - 1:
            print(f'  Ép {epoch+1:3d}/{EPOCHS} | loss={loss:.4f} | acc={acc:.2f}% | '
                  f'gain={gain}% | {dt:.1f}s/ép')

    final_acc  = history[-1]['acc']
    mean_gain  = np.mean([r['gain'] for r in history if r['gain'] > 0])
    mean_time  = np.mean([r['time'] for r in history])
    return {'variant': variant, 'acc': final_acc, 'gain': mean_gain,
            'time_ep': mean_time, 'history': history, 'K': K_current, 'N': N}

In [ ]:
# ── Lancer les expériences ────────────────────────────────────────────────
print('='*60)
print('EXPÉRIENCE 1 — SGD standard (référence)')
print('='*60)
res_std = run_experiment('standard')

print('='*60)
print('EXPÉRIENCE 2 — K-ABENA N=0 (mode agressif)')
print('Différence vs standard : +2 lignes dans la boucle')
print('='*60)
res_ka0 = run_experiment('ka_fixed_n0', K=None, N=0.0)

print('='*60)
print('EXPÉRIENCE 3 — K-ABENA N=0.4 (mode modéré)')
print('='*60)
res_ka4 = run_experiment('ka_fixed_n4', K=None, N=0.4)

print('='*60)
print('EXPÉRIENCE 4 — K-ABENA Adaptatif (warm-up)')
print('='*60)
res_adp = run_experiment('adaptive', K=None, N=0.3)

print('='*60)
print('EXPÉRIENCE 5 — Adam + K-ABENA')
print('='*60)
res_adam = run_experiment('adam_ka', K=None, N=0.3, optimizer_cls=torch.optim.Adam)

In [ ]:
# ── Tableau comparatif final ──────────────────────────────────────────────
import pandas as pd

targets = {'standard': 93.2, 'ka_fixed_n0': 94.2, 'ka_fixed_n4': 94.6,
           'adaptive': 94.9, 'adam_ka': 95.1}

rows = []
for res in [res_std, res_ka0, res_ka4, res_adp, res_adam]:
    v = res['variant']
    rows.append({
        'Variante':       v,
        'Acc. obtenue':   f"{res['acc']:.2f}%",
        'Acc. cible':     f"{targets[v]:.1f}%",
        'Δ':              f"{res['acc'] - targets[v]:+.2f}%",
        'Gain comp.':     f"{res['gain']:.1f}%" if res['gain'] > 0 else '0%',
        'Temps/ép.':      f"{res['time_ep']:.1f}s",
    })

df = pd.DataFrame(rows)
print('\n' + '='*70)
print('RÉSULTATS FINAUX CIFAR-10 (PyTorch) — Comparaison avec Ch.14')
print('='*70)
print(df.to_string(index=False))
print('\nNote : Δ = différence avec la cible du livre (peut varier selon GPU/seed/époques).')

In [ ]:
# ── Visualisation ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
all_results = [('SGD standard', res_std, '#B4B2A9', '--'),
               ('K-ABENA N=0',  res_ka0, '#E24B4A', '-'),
               ('K-ABENA N=0.4',res_ka4, '#EF9F27', '-'),
               ('K-ABENA Adp.', res_adp, '#1DD0FF', '-'),
               ('Adam+K-ABENA', res_adam,'#1D9E75', '-')]

for name, res, col, ls in all_results:
    ep   = [r['epoch']+1 for r in res['history']]
    accs = [r['acc']     for r in res['history']]
    axes[0].plot(ep, accs, color=col, lw=2.2, ls=ls, label=name)

axes[0].set_xlabel('Époque'); axes[0].set_ylabel('Top-1 Accuracy (%)')
axes[0].set_title('CIFAR-10 — Convergence (PyTorch)')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.2)

for name, res, col, ls in all_results[1:]:  # sans standard
    ep    = [r['epoch']+1  for r in res['history']]
    gains = [r['gain']     for r in res['history']]
    axes[1].plot(ep, gains, color=col, lw=2.2, ls=ls, label=name)

axes[1].set_xlabel('Époque'); axes[1].set_ylabel('Gain computationnel (%)')
axes[1].set_title('CIFAR-10 — Gain computationnel K-ABENA')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.2)

plt.suptitle('Reproduction Ch.14 — CIFAR-10 PyTorch', fontweight='bold')
plt.tight_layout()
plt.savefig('ch14_cifar10_pytorch_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée : ch14_cifar10_pytorch_results.png')